# KITTI Depth Completion: download everything into Google Drive and unzip

What this notebook does:
1. Downloads **61 zips** from the official KITTI S3 mirror (no login needed):
   the 3 depth zips + 58 raw zips that hold the **RGB** of our Train / Val drives.
2. Saves every zip in `MyDrive/KITTI_DC/zips/`.
3. Unzips only what we use into `MyDrive/KITTI_DC/`.
4. Writes `train_list.txt`, `val_list.txt`, `test_list.txt` (RGB, LiDAR, GT path per line) and checks every file exists.

**Google Drive space needed: about 70 GB** (about 60 GB of zips + about 10 GB unzipped).
Safe to re-run: finished zips and already unzipped files are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, urllib.request

ROOT  = '/content/drive/MyDrive/KITTI_DC'   # final data lives here
ZIPS  = f'{ROOT}/zips'                      # all downloaded zips
LOCAL = '/content/dl'                       # Colab local disk, used only while downloading
os.makedirs(ZIPS, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

print(f"Google Drive free space (as reported): {shutil.disk_usage('/content/drive/MyDrive').free / 1e9:.0f} GB, need about 70 GB")

In [ ]:
BASE = 'https://s3.eu-central-1.amazonaws.com/avg-kitti/'

TRAIN = [
    '2011_09_26_drive_0017',
    '2011_09_26_drive_0027',
    '2011_09_26_drive_0032',
    '2011_09_26_drive_0039',
    '2011_09_26_drive_0056',
    '2011_09_26_drive_0086',
    '2011_09_26_drive_0101',
    '2011_09_28_drive_0001',
    '2011_09_28_drive_0045',
    '2011_09_28_drive_0054',
    '2011_09_28_drive_0068',
    '2011_09_28_drive_0070',
    '2011_09_28_drive_0078',
    '2011_09_28_drive_0090',
    '2011_09_28_drive_0094',
    '2011_09_28_drive_0095',
    '2011_09_28_drive_0096',
    '2011_09_28_drive_0098',
    '2011_09_28_drive_0100',
    '2011_09_28_drive_0104',
    '2011_09_28_drive_0113',
    '2011_09_28_drive_0117',
    '2011_09_28_drive_0126',
    '2011_09_28_drive_0135',
    '2011_09_28_drive_0138',
    '2011_09_28_drive_0143',
    '2011_09_28_drive_0145',
    '2011_09_28_drive_0155',
    '2011_09_28_drive_0165',
    '2011_09_28_drive_0167',
    '2011_09_28_drive_0183',
    '2011_09_28_drive_0184',
    '2011_09_28_drive_0192',
    '2011_09_28_drive_0195',
    '2011_09_28_drive_0199',
    '2011_09_28_drive_0204',
    '2011_09_28_drive_0205',
    '2011_09_28_drive_0214',
    '2011_09_28_drive_0216',
    '2011_09_28_drive_0220',
    '2011_09_30_drive_0027',
    '2011_09_30_drive_0033',
    '2011_10_03_drive_0042',
]

VAL = [
    '2011_09_26_drive_0001',
    '2011_09_26_drive_0011',
    '2011_09_26_drive_0018',
    '2011_09_26_drive_0048',
    '2011_09_26_drive_0052',
    '2011_09_26_drive_0060',
    '2011_09_28_drive_0057',
    '2011_09_28_drive_0065',
    '2011_09_28_drive_0066',
    '2011_09_28_drive_0089',
    '2011_09_28_drive_0102',
    '2011_09_28_drive_0122',
    '2011_09_28_drive_0136',
    '2011_09_28_drive_0156',
    '2011_09_28_drive_0208',
]

FILES = {
    'data_depth_annotated.zip': BASE + 'data_depth_annotated.zip',   # GT depth      (14.2 GB)
    'data_depth_velodyne.zip':  BASE + 'data_depth_velodyne.zip',    # sparse LiDAR  (5.2 GB)
    'data_depth_selection.zip': BASE + 'data_depth_selection.zip',   # Test set      (2.0 GB)
}
for d in TRAIN + VAL:                                                 # RGB of Train / Val drives
    FILES[f'{d}_sync.zip'] = f'{BASE}raw_data/{d}/{d}_sync.zip'

print(len(TRAIN), 'train drives,', len(VAL), 'val drives,', len(FILES), 'zips')

## 1. Download all zips into Google Drive

In [ ]:
subprocess.run('apt-get -qq install -y aria2 > /dev/null', shell=True, check=True)

def remote_size(url):
    return int(urllib.request.urlopen(urllib.request.Request(url, method='HEAD')).headers['Content-Length'])

total = len(FILES)
for i, (name, url) in enumerate(FILES.items(), 1):
    dst, size = f'{ZIPS}/{name}', remote_size(url)
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        print(f'[{i}/{total}] already in Drive: {name}')
        continue
    print(f'[{i}/{total}] downloading {name} ({size / 1e9:.2f} GB)')
    subprocess.run(['aria2c', '-q', '-c', '-x', '8', '-s', '8', '--file-allocation=none',
                    '-d', LOCAL, '-o', name, url], check=True)
    assert os.path.getsize(f'{LOCAL}/{name}') == size, f'size mismatch: {name}'
    shutil.move(f'{LOCAL}/{name}', dst)          # copy to Drive, free local disk

print('all zips in', ZIPS)

## 2. Unzip only what we use
- `data_depth_selection.zip` → `val_selection_cropped` (Test: RGB + LiDAR + GT)
- `data_depth_annotated.zip` → GT of the 58 Train / Val drives, camera `image_02`
- `data_depth_velodyne.zip` → LiDAR of the same drives
- each `<drive>_sync.zip` → only the RGB folder `image_02/data`

Writing many small files to Google Drive is slow; this cell can take a while. Re-running skips files already unzipped.

In [ ]:
def unzip(zip_name, patterns, dest):
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['unzip', '-q', '-n', f'{ZIPS}/{zip_name}', *patterns, '-d', dest], check=True)

ALL = TRAIN + VAL

print('Test set ...')
unzip('data_depth_selection.zip', [f'depth_selection/val_selection_cropped/{k}/*' for k in ('image', 'velodyne_raw', 'groundtruth_depth')], ROOT)

print('GT depth ...')
unzip('data_depth_annotated.zip', [f'train/{d}_sync/proj_depth/groundtruth/image_02/*' for d in ALL], ROOT)

print('sparse LiDAR ...')
unzip('data_depth_velodyne.zip', [f'train/{d}_sync/proj_depth/velodyne_raw/image_02/*' for d in ALL], ROOT)

for i, d in enumerate(ALL, 1):
    print(f'RGB [{i}/{len(ALL)}] {d}')
    unzip(f'{d}_sync.zip', [f'{d[:10]}/{d}_sync/image_02/data/*'], f'{ROOT}/raw')

print('done')

## 3. Build the pair lists and check every file exists
Each line: `RGB path  LiDAR path  GT path`. Expected: Train 8237, Val 1023, Test 1000.

In [ ]:
def files(folder):
    return set(os.listdir(folder)) if os.path.isdir(folder) else set()

os.makedirs(f'{ROOT}/splits', exist_ok=True)

def build(split, drives):
    lines, missing = [], 0
    for d in drives:
        rgb_dir = f'{ROOT}/raw/{d[:10]}/{d}_sync/image_02/data'
        lid_dir = f'{ROOT}/train/{d}_sync/proj_depth/velodyne_raw/image_02'
        gt_dir  = f'{ROOT}/train/{d}_sync/proj_depth/groundtruth/image_02'
        rgb, lid = files(rgb_dir), files(lid_dir)
        for f in sorted(files(gt_dir)):                  # GT decides which frames are used
            if f in rgb and f in lid:
                lines.append(f'{rgb_dir}/{f} {lid_dir}/{f} {gt_dir}/{f}')
            else:
                missing += 1
    return lines, missing

T = f'{ROOT}/depth_selection/val_selection_cropped'
def build_test():
    lines, missing = [], 0
    lid, gt = files(f'{T}/velodyne_raw'), files(f'{T}/groundtruth_depth')
    for f in sorted(files(f'{T}/image')):
        fl = f.replace('_sync_image_', '_sync_velodyne_raw_', 1)
        fg = f.replace('_sync_image_', '_sync_groundtruth_depth_', 1)
        if fl in lid and fg in gt:
            lines.append(f'{T}/image/{f} {T}/velodyne_raw/{fl} {T}/groundtruth_depth/{fg}')
        else:
            missing += 1
    return lines, missing

for split, (lines, missing) in [('train', build('train', TRAIN)), ('val', build('val', VAL)), ('test', build_test())]:
    with open(f'{ROOT}/splits/{split}_list.txt', 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f'{split:5s}: {len(lines)} pairs, {missing} incomplete  ->  {ROOT}/splits/{split}_list.txt')
    print('   example:', lines[0].replace(' ', '\n            ') if lines else 'NONE')

## 4. Finish uploading to Google Drive
Colab writes to a local cache and uploads to Drive in the background. This cell waits until **every file is uploaded**.
Do not close the tab or stop the runtime before it prints `all files are in Google Drive`.

In [ ]:
drive.flush_and_unmount()
print('all files are in Google Drive')